In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# ==========================================
# 1. RESIDUAL BLOCK DEFINITION
# ==========================================
def residual_block(x, filters, kernel_size=3):
    """
    This is the secret weapon of ResNet. 
    It creates a 'Skip Connection' that adds the original input 
    back to the output of the convolutional layers, preventing data loss.
    """
    # Shortcut (the skip connection)
    shortcut = x

    # First Convolution
    x = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Convolution
    x = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)

    # If the number of filters changed, adjust the shortcut to match
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding='same')(shortcut)

    # Add the shortcut back to the main path!
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

In [3]:
# ==========================================
# 2. LOAD THE DATA
# ==========================================
print("Loading Deep Learning Dataset (Pure Signal)...")
X_signals = np.load("X_signals.npy")  # Shape: (2150, 100, 2)
y_glucose = np.load("y_glucose.npy")  # Shape: (2150,)

X_train, X_test, y_train, y_test = train_test_split(
    X_signals, y_glucose, test_size=0.2, random_state=42
)

Loading Deep Learning Dataset (Pure Signal)...


In [4]:
# ==========================================
# 3. BUILD THE RESNET-1D ARCHITECTURE
# ==========================================
input_signal = layers.Input(shape=(100, 2), name="signal_input")

# Initial smooth convolution
x = layers.Conv1D(32, kernel_size=7, padding='same')(input_signal)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling1D(pool_size=2)(x)

# Stack of Residual Blocks (The deep learning engine)
x = residual_block(x, filters=32)
x = residual_block(x, filters=64)
x = layers.MaxPooling1D(pool_size=2)(x)
x = residual_block(x, filters=128)

# Global Average Pooling (Better than Flattening for ResNets)
x = layers.GlobalAveragePooling1D()(x)

# Final decision layers
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='linear', name="glucose_output")(x)

# Compile Model
model = Model(inputs=input_signal, outputs=output)

# We use a slightly smaller learning rate because ResNet is highly sensitive
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

In [5]:
# ==========================================
# 4. TRAIN THE MODEL
# ==========================================
print("\nTraining the ResNet-1D Model...")

# Reduce learning rate if the model gets stuck on a plateau
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history = model.fit(
    x=X_train, 
    y=y_train,
    validation_data=(X_test, y_test),
    epochs=150,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


Training the ResNet-1D Model...
Epoch 1/150


54/54 ━━━━━━━━━━━━━━━━━━━━ 19s 155ms/step - loss: 9649.0127 - mae: 88.6975 - val_loss: 10990.7910 - val_mae: 98.8913 - learning_rate: 5.0000e-04
Epoch 2/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1523.8143 - mae: 30.8810 - val_loss: 5819.3633 - val_mae: 67.8578 - learning_rate: 5.0000e-04
Epoch 3/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1398.9963 - mae: 29.4075 - val_loss: 3805.3379 - val_mae: 50.9636 - learning_rate: 5.0000e-04
Epoch 4/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1331.0886 - mae: 28.9737 - val_loss: 2164.8201 - val_mae: 33.4712 - learning_rate: 5.0000e-04
Epoch 5/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1348.4301 - mae: 28.6409 - val_loss: 1638.5651 - val_mae: 28.3426 - learning_rate: 5.0000e-04
Epoch 6/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1262.2006 - mae: 28.0079 - val_loss: 2080.8950 - val_mae: 32.8707 - learning_rate: 5.0000e-04
Epoch 7/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1268.6567 - mae: 28.0478 - val_

In [6]:
# ==========================================
# 5. EVALUATE
# ==========================================
print("\n==================================================")
print("🏆 RESNET-1D RESULTS (PURE SIGNAL)")
print("==================================================")

predictions = model.predict(X_test).flatten()
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"➡️ Mean Absolute Error (MAE): {mae:.2f} mg/dL")
print(f"➡️ Root Mean Squared Error (RMSE): {rmse:.2f} mg/dL")
print(f"➡️ R-Squared (R2):            {r2:.2f}")


🏆 RESNET-1D RESULTS (PURE SIGNAL)
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step 
➡️ Mean Absolute Error (MAE): 21.96 mg/dL
➡️ Root Mean Squared Error (RMSE): 29.24 mg/dL
➡️ R-Squared (R2):            0.30


In [7]:
model.save("One_D_Resnet.h5")
print("\nModel saved successfully as 'One_D_Resnet.h5'")


Model saved successfully as 'One_D_Resnet.h5'
